In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/online_retail.csv"
)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Revenue"] = df["Quantity"] * df["UnitPrice"]

identified = df[df["CustomerID"] != "0"].copy()


customer_summary = (
    identified
    .groupby("CustomerID")
    .agg(
        Purchases=("InvoiceNo", "nunique"),
        Total_Spend=("Revenue", "sum"),
        Avg_Transaction=("Revenue", "mean"),
        Unique_Products=("StockCode", "nunique"),
    )
)

(customer_summary.index == "0").sum()


customer_purchases = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)
customer_purchases["Days_Between"] = (
    customer_purchases
    .groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)
# (customer_purchases["CustomerID"] == "0").sum()
(customer_purchases["CustomerID"] == 0).sum()


purchase_timing = (
    customer_purchases
    .groupby("CustomerID")
    .agg(
        Avg_Days_Between=("Days_Between", "mean"),
        Variation_Days_Between=("Days_Between", "std")
    )
)

customer_summary = customer_summary.join(purchase_timing)

customer_count = len(customer_summary)

top_20_count = round(customer_count * 0.20)

top_20_customers = (
    customer_summary
    .sort_values("Total_Spend", ascending=False)
    .head(top_20_count)
    .copy()
)

other_80_customers = (
    customer_summary
    .drop(top_20_customers.index)
    .copy()
)


/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_45050/739893729.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


#### Digging deeper into the number of days between purchases for all customers

In [15]:
customer_last_purchase = (
    identified
    .groupby("CustomerID")["InvoiceDate"]
    .max()
    .rename("Last_Purchase")
)

analysis_date = identified["InvoiceDate"].max()

customer_summary = customer_summary.join(customer_last_purchase)

customer_summary["Days_Since_Last_Purchase"] = (
    analysis_date - customer_summary["Last_Purchase"]
).dt.days

In [16]:
customer_summary["Days_Since_Last_Purchase"].describe().round(2)

count    4372.00
mean       91.05
std       100.77
min         0.00
25%        16.00
50%        49.00
75%       142.00
max       373.00
Name: Days_Since_Last_Purchase, dtype: float64

In [17]:
days_since_last_comparison = pd.DataFrame({
    "High-Value": customer_summary[
        customer_summary.index.isin(top_20_customers.index)
    ]["Days_Since_Last_Purchase"].describe(),

    "Other 80%": customer_summary[
        customer_summary.index.isin(other_80_customers.index)
    ]["Days_Since_Last_Purchase"].describe()
}).round(2)

days_since_last_comparison

,High-Value,Other 80%
count,874.00,3498.00
mean,29.54,106.41
std,44.64,104.94
min,0.00,0.00
25%,4.00,23.00
50%,14.50,63.00
75%,34.00,174.75
max,319.00,373.00


In [18]:
customer_purchases["Days_Between"].describe().round(2)

count    17818.00
mean        32.81
std         48.62
min          0.00
25%          3.19
50%         14.05
75%         41.16
max        365.98
Name: Days_Between, dtype: float64

In [19]:
customer_purchases["Days_Between"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)

0.25      3.19
0.50     14.05
0.75     41.16
0.90     88.92
0.95    131.02
0.99    240.78
Name: Days_Between, dtype: float64

In [20]:
high_value_ids = top_20_customers.index

high_value_intervals = customer_purchases[
    customer_purchases["CustomerID"].isin(high_value_ids)
]["Days_Between"].dropna()

other_intervals = customer_purchases[
    customer_purchases["CustomerID"].isin(other_80_customers.index)
]["Days_Between"].dropna()

interval_comparison = pd.DataFrame({
    "High-Value": high_value_intervals.describe(),
    "Other 80%": other_intervals.describe()
}).round(2)

interval_comparison

,High-Value,Other 80%
count,11566.00,6252.00
mean,20.63,55.35
std,28.98,66.34
min,0.00,0.00
25%,2.01,7.23
50%,10.00,31.06
75%,28.01,78.11
max,337.09,365.98


In [21]:
interval_percentiles = pd.DataFrame({
    "High-Value": high_value_intervals.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ),

    "Other 80%": other_intervals.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
}).round(2)

interval_percentiles

,High-Value,Other 80%
0.25,2.01,7.23
0.50,10.00,31.06
0.75,28.01,78.11
0.90,54.15,147.00
0.95,76.10,196.80
0.99,132.89,307.01


In [22]:
customer_summary["Variation_Days_Between"]

CustomerID
12346.0          NaN
12347.0    18.400530
12348.0    69.952924
12349.0          NaN
12350.0          NaN
             ...    
18280.0          NaN
18281.0          NaN
18282.0    78.340557
18283.0    18.609339
18287.0    89.821221
Name: Variation_Days_Between, Length: 4372, dtype: float64

In [23]:
rhythm_variation = pd.DataFrame({
    "High-Value": customer_summary[
        customer_summary.index.isin(top_20_customers.index)
    ]["Variation_Days_Between"].dropna().describe(),

    "Other 80%": customer_summary[
        customer_summary.index.isin(other_80_customers.index)
    ]["Variation_Days_Between"].dropna().describe()
}).round(2)

rhythm_variation

,High-Value,Other 80%
count,845.00,1397.00
mean,30.03,51.51
std,23.00,42.25
min,0.00,0.00
25%,15.16,21.85
50%,24.58,40.59
75%,37.75,69.95
max,190.01,253.25


In [24]:
customer_summary["Expected_Next_Purchase"] = (
    customer_summary["Last_Purchase"]
    + pd.to_timedelta(
        customer_summary["Avg_Days_Between"],
        unit="D"
    )
)

In [25]:
customer_summary["Days_Past_Average"] = (
    analysis_date - customer_summary["Expected_Next_Purchase"]
).dt.days

In [26]:
customer_summary["Interval_Z_Score"] = (
    customer_summary["Days_Past_Average"]
    / customer_summary["Variation_Days_Between"]
)

In [27]:
late_customers = customer_summary[
    (customer_summary["Purchases"] > 1) &
    (customer_summary["Days_Past_Average"] > 0)
]

print("Late customers:", len(late_customers))

Late customers: 1155


In [28]:
repeat_customers = customer_summary[
    customer_summary["Purchases"] > 1
]

late_percentage = (
    len(late_customers)
    / len(repeat_customers)
    * 100
)

print("Late repeat customers:", round(late_percentage, 2), "%")

Late repeat customers: 37.76 %


In [29]:
repeat_customer_summary = customer_summary[
    customer_summary["Purchases"] > 1
].copy()

late_customer_levels = pd.Series({
    "Past Average": (
        repeat_customer_summary["Days_Past_Average"] > 0
    ).sum(),

    "1+ SD Late": (
        repeat_customer_summary["Interval_Z_Score"] >= 1
    ).sum(),

    "2+ SD Late": (
        repeat_customer_summary["Interval_Z_Score"] >= 2
    ).sum(),

    "3+ SD Late": (
        repeat_customer_summary["Interval_Z_Score"] >= 3
    ).sum()
})

late_customer_levels

Past Average    1155
1+ SD Late       431
2+ SD Late       310
3+ SD Late       237
dtype: int64

In [30]:
high_value_late = repeat_customer_summary[
    repeat_customer_summary.index.isin(top_20_customers.index) &
    (repeat_customer_summary["Days_Past_Average"] > 0)
].copy()

In [31]:
high_value_late[
    [
        "Purchases",
        "Total_Spend",
        "Avg_Days_Between",
        "Variation_Days_Between",
        "Days_Since_Last_Purchase",
        "Days_Past_Average",
        "Interval_Z_Score"
    ]
].sort_values(
    "Days_Past_Average",
    ascending=False
).head(20)

,Purchases,Total_Spend,Avg_Days_Between,Variation_Days_Between,Days_Since_Last_Purchase,Days_Past_Average,Interval_Z_Score
CustomerID,,,,,,,
17850.0,35,5288.630,2.095833,11.993274,301,299.0,24.930640
12501.0,2,2089.680,21.059722,NaN,314,293.0,NaN
15808.0,5,3724.770,15.765278,14.607794,305,290.0,19.852416
13093.0,13,7741.470,8.841204,8.731641,266,257.0,29.433186
17230.0,11,3466.670,9.520139,13.464336,263,254.0,18.864651
15032.0,5,4464.100,27.526562,55.025348,255,228.0,4.143545
12754.0,5,2949.120,18.234201,21.382066,235,216.0,10.101924
12755.0,4,2203.200,38.659259,42.037429,249,210.0,4.995548
15235.0,12,2247.510,14.152904,11.456529,217,202.0,17.631867


In [32]:
# new customers

new_customers = customer_summary[
    customer_summary["Purchases"] == 1
].copy()

In [33]:
new_customer_timing = pd.DataFrame({
    "All One-Purchase": new_customers["Days_Since_Last_Purchase"].describe(),

    "Nov/Dec 2011": new_customers[
        new_customers["Last_Purchase"].dt.year.eq(2011) &
        new_customers["Last_Purchase"].dt.month.isin([11, 12])
    ]["Days_Since_Last_Purchase"].describe(),

    "Earlier": new_customers[
        ~(
            new_customers["Last_Purchase"].dt.year.eq(2011) &
            new_customers["Last_Purchase"].dt.month.isin([11, 12])
        )
    ]["Days_Since_Last_Purchase"].describe()
}).round(2)

new_customer_timing

,All One-Purchase,Nov/Dec 2011,Earlier
count,1313.00,250.00,1063.00
mean,156.61,20.06,188.73
std,117.64,10.31,107.94
min,0.00,0.00,38.00
25%,50.00,11.00,78.00
50%,131.00,21.00,185.00
75%,262.00,29.00,280.00
max,373.00,38.00,373.00


In [34]:
customer_summary["Purchases"].value_counts().sort_index()

Purchases
1      1313
2       817
3       490
4       377
5       288
       ... 
118       2
128       1
169       1
224       1
248       1
Name: count, Length: 65, dtype: int64

In [35]:
customer_summary[
    customer_summary["Purchases"] >= 3
]["Purchases"].describe().round(2)

count    2242.00
mean        8.58
std        12.03
min         3.00
25%         4.00
50%         5.00
75%         9.00
max       248.00
Name: Purchases, dtype: float64

In [36]:
high_value_rhythm = (
    customer_purchases[
        customer_purchases["CustomerID"].isin(top_20_customers.index)
    ]
    .groupby("CustomerID")["Days_Between"]
    .agg(
        Purchases="count",
        Avg_Days="mean",
        Median_Days="median",
        Min_Days="min",
        Max_Days="max",
        Variation_Days="std"
    )
    .sort_values("Avg_Days")
)

high_value_rhythm.head(20)

,Purchases,Avg_Days,Median_Days,Min_Days,Max_Days,Variation_Days
CustomerID,,,,,,
16000.0,2,0.001042,0.001042,0.000694,0.001389,0.000491
17084.0,1,0.003472,0.003472,0.003472,0.003472,NaN
18139.0,7,0.155952,0.045833,0.011111,0.706250,0.253096
17509.0,10,0.595972,0.033333,0.000694,2.730556,1.129189
14911.0,247,1.506379,0.836806,0.000000,15.768750,2.249975
12748.0,223,1.672559,0.759028,0.000694,30.185417,3.139381
17850.0,34,2.095833,0.000694,0.000000,69.965972,11.993274
17841.0,168,2.213695,2.109375,0.000000,15.952778,2.111138
14606.0,127,2.929960,2.886111,0.000694,12.896528,2.501786


In [37]:
high_value_rhythm.describe().round(2)

,Purchases,Avg_Days,Median_Days,Min_Days,Max_Days,Variation_Days
count,874.00,861.00,861.00,861.00,861.00,845.00
mean,13.23,35.12,28.82,7.69,86.10,30.03
std,17.60,29.15,30.40,24.71,49.22,23.00
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,5.00,17.50,11.05,0.00,50.17,15.16
50%,9.00,28.46,19.90,0.01,77.02,24.58
75%,15.00,43.19,36.57,4.18,109.09,37.75
max,247.00,337.09,337.09,337.09,337.09,190.01


In [38]:
customer_summary["Recency_vs_Rhythm"] = (
    customer_summary["Days_Since_Last_Purchase"]
    / customer_summary["Avg_Days_Between"].replace(0, np.nan)
)

recency_vs_rhythm = customer_summary[
    customer_summary["Purchases"] > 1
]["Recency_vs_Rhythm"]

recency_vs_rhythm.describe().round(2)

count      3056.00
mean       1146.93
std       14215.23
min           0.00
25%           0.21
50%           0.64
75%           1.95
max      383040.00
Name: Recency_vs_Rhythm, dtype: float64

#### Do short-duration/one-day products help maintain customer usual purchasing cadence?

In [43]:
product_date_range = (
    identified
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Min_Date=("InvoiceDate", "min"),
        Max_Date=("InvoiceDate", "max")
    )
    .reset_index()
)

product_date_range["Days_Active"] = (
    product_date_range["Max_Date"] - product_date_range["Min_Date"]
).dt.days

products_within_30_days = (
    product_date_range[
        product_date_range["Days_Active"] <= 30
    ]
    .sort_values("Days_Active")
)
zero_day_products = set(
    product_date_range.loc[
        product_date_range["Days_Active"] == 0,
        "StockCode"
    ]
)

zero_day_stock_codes = set(zero_day_products)

customer_purchase_events = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])
    .agg(
        Purchase_Date=("InvoiceDate", "min"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

zero_day_invoice = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])["StockCode"]
    .apply(lambda x: x.isin(zero_day_stock_codes).any())
    .rename("Contains_Zero_Day_Product")
    .reset_index()
)

customer_purchase_events = customer_purchase_events.merge(
    zero_day_invoice,
    on=["CustomerID", "InvoiceNo"],
    how="left"
)

customer_purchase_events = customer_purchase_events.sort_values(
    ["CustomerID", "Purchase_Date"]
)

customer_purchase_events["Days_Between"] = (
    customer_purchase_events
    .groupby("CustomerID")["Purchase_Date"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

In [45]:
zero_day_interval_comparison = pd.DataFrame({
    "After 0-Day Purchase": customer_purchase_events[
        customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_Between"].dropna().describe(),

    "After Regular Purchase": customer_purchase_events[
        ~customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_Between"].dropna().describe()
}).round(2)

In [46]:
zero_day_stock_codes = set(zero_day_products)

customer_purchase_events = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])
    .agg(
        Purchase_Date=("InvoiceDate", "min"),
        Revenue=("Revenue", "sum")
    )
    .reset_index()
)

zero_day_invoice = (
    identified
    .groupby(["CustomerID", "InvoiceNo"])["StockCode"]
    .apply(lambda x: x.isin(zero_day_stock_codes).any())
    .rename("Contains_Zero_Day_Product")
    .reset_index()
)

customer_purchase_events = customer_purchase_events.merge(
    zero_day_invoice,
    on=["CustomerID", "InvoiceNo"],
    how="left"
)

customer_purchase_events = customer_purchase_events.sort_values(
    ["CustomerID", "Purchase_Date"]
)

customer_purchase_events["Days_Between"] = (
    customer_purchase_events
    .groupby("CustomerID")["Purchase_Date"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

In [47]:
zero_day_interval_comparison = pd.DataFrame({
    "0-Day Purchase": customer_purchase_events[
        customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_Between"].dropna().describe(),

    "Regular Purchase": customer_purchase_events[
        ~customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_Between"].dropna().describe()
}).round(2)

zero_day_interval_comparison

,0-Day Purchase,Regular Purchase
count,13447.00,4371.00
mean,37.46,18.51
std,50.85,37.59
min,0.00,0.00
25%,5.23,0.18
50%,19.80,5.98
75%,48.16,17.79
max,365.98,364.92


In [48]:
customer_purchase_events["Days_To_Next_Purchase"] = (
    customer_purchase_events
    .groupby("CustomerID")["Purchase_Date"]
    .shift(-1)
    - customer_purchase_events["Purchase_Date"]
).dt.total_seconds().div(86400)

In [49]:
next_purchase_comparison = pd.DataFrame({
    "After 0-Day Purchase": customer_purchase_events[
        customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_To_Next_Purchase"].dropna().describe(),

    "After Regular Purchase": customer_purchase_events[
        ~customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Days_To_Next_Purchase"].dropna().describe()
}).round(2)

next_purchase_comparison

,After 0-Day Purchase,After Regular Purchase
count,13873.00,3945.00
mean,33.51,30.34
std,48.69,48.32
min,0.00,0.00
25%,3.96,1.97
50%,14.97,11.91
75%,42.06,36.76
max,365.98,356.05


In [50]:
valid_intervals = customer_purchase_events[
    customer_purchase_events["Days_Between"] >= 1
]

customer_rhythm = (
    valid_intervals
    .groupby("CustomerID")["Days_Between"]
    .agg(
        Typical_Interval="median",
        Average_Interval="mean",
        Interval_Variation="std",
        Interval_Count="count"
    )
)

In [51]:
customer_purchase_events = customer_purchase_events.merge(
    customer_rhythm,
    on="CustomerID",
    how="left"
)

In [52]:
customer_purchase_events["Next_Purchase_vs_Rhythm"] = (
    customer_purchase_events["Days_To_Next_Purchase"]
    / customer_purchase_events["Typical_Interval"]
)

In [53]:
rhythm_comparison = pd.DataFrame({
    "After 0-Day Purchase": customer_purchase_events[
        customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Next_Purchase_vs_Rhythm"].dropna().describe(),

    "After Regular Purchase": customer_purchase_events[
        ~customer_purchase_events["Contains_Zero_Day_Product"]
    ]["Next_Purchase_vs_Rhythm"].dropna().describe()
}).round(2)

rhythm_comparison

,After 0-Day Purchase,After Regular Purchase
count,13794.00,3921.00
mean,1.09,1.25
std,1.47,2.53
min,0.00,0.00
25%,0.27,0.17
50%,0.93,0.88
75%,1.31,1.43
max,42.61,60.45


In [54]:
customer_purchase_events["Next_Purchase_Timing"] = pd.cut(
    customer_purchase_events["Next_Purchase_vs_Rhythm"],
    bins=[0, 0.75, 1.25, 2, np.inf],
    labels=[
        "Earlier Than Usual",
        "Around Normal",
        "Somewhat Late",
        "Very Late"
    ]
)

In [55]:
rhythm_timing_comparison = pd.crosstab(
    customer_purchase_events["Contains_Zero_Day_Product"],
    customer_purchase_events["Next_Purchase_Timing"],
    normalize="index"
) * 100

rhythm_timing_comparison.round(2)

Next_Purchase_Timing,Earlier Than Usual,Around Normal,Somewhat Late,Very Late
Contains_Zero_Day_Product,,,,
False,45.01,25.08,15.05,14.86
True,42.82,30.15,15.32,11.72


In [56]:
customer_purchase_events["Days_Past_Rhythm"] = (
    customer_purchase_events["Days_Between"]
    - customer_purchase_events["Typical_Interval"]
)

customer_purchase_events["Late_Ratio"] = (
    customer_purchase_events["Days_Between"]
    / customer_purchase_events["Typical_Interval"]
)

In [57]:
late_zero_day = customer_purchase_events[
    (customer_purchase_events["Contains_Zero_Day_Product"]) &
    (customer_purchase_events["Late_Ratio"] >= 1.25)
].copy()

In [58]:
late_zero_day.shape

(4194, 15)

In [59]:
late_zero_day[
    [
        "CustomerID",
        "Purchase_Date",
        "Days_Between",
        "Typical_Interval",
        "Late_Ratio"
    ]
].head(10)

,CustomerID,Purchase_Date,Days_Between,Typical_Interval,Late_Ratio
7,12347.0,2011-10-31 12:25:00,90.150694,58.460069,1.542090
23,12352.0,2011-09-20 14:34:00,181.934722,14.552778,12.501718
31,12356.0,2011-11-17 08:40:00,222.838194,151.475694,1.471115
39,12359.0,2011-10-13 12:47:00,132.014583,50.950000,2.591061
47,12362.0,2011-04-28 09:12:00,62.838194,14.035764,4.477006
48,12362.0,2011-07-07 12:32:00,70.138889,14.035764,4.997155
49,12362.0,2011-08-11 15:02:00,35.104167,14.035764,2.501051
51,12362.0,2011-09-28 12:04:00,40.059722,14.035764,2.854118
56,12362.0,2011-11-28 14:55:00,24.241667,14.035764,1.727136
71,12370.0,2011-10-19 14:51:00,223.085417,83.131944,2.683510


In [60]:
late_zero_day["Next_Purchase_vs_Rhythm"] = (
    late_zero_day["Days_To_Next_Purchase"]
    / late_zero_day["Typical_Interval"]
)

In [61]:
late_zero_day["Next_Purchase_Timing"] = pd.cut(
    late_zero_day["Next_Purchase_vs_Rhythm"],
    bins=[0, 0.75, 1.25, 2, np.inf],
    labels=[
        "Earlier Than Usual",
        "Around Normal",
        "Somewhat Late",
        "Very Late"
    ]
)

In [62]:
late_zero_day["Next_Purchase_Timing"].value_counts(
    normalize=True
).mul(100).round(2)

Next_Purchase_Timing
Earlier Than Usual    49.31
Around Normal         25.09
Somewhat Late         13.14
Very Late             12.46
Name: proportion, dtype: float64

In [63]:
late_zero_day_comparison = pd.DataFrame({
    "Before 0-Day Purchase": late_zero_day["Late_Ratio"].describe(),
    "After 0-Day Purchase": late_zero_day["Next_Purchase_vs_Rhythm"].describe()
}).round(2)

late_zero_day_comparison

,Before 0-Day Purchase,After 0-Day Purchase
count,4194.00,3721.00
mean,2.70,1.02
std,2.86,1.34
min,1.25,0.00
25%,1.52,0.23
50%,1.91,0.75
75%,2.82,1.26
max,60.45,29.47


In [64]:
late_customer_ids = late_zero_day["CustomerID"].unique()

late_customer_history = (
    customer_purchase_events[
        customer_purchase_events["CustomerID"].isin(late_customer_ids)
    ]
    [
        [
            "CustomerID",
            "Purchase_Date",
            "Days_Between",
            "Typical_Interval",
            "Contains_Zero_Day_Product",
            "Days_To_Next_Purchase",
            "Next_Purchase_vs_Rhythm"
        ]
    ]
    .sort_values(["CustomerID", "Purchase_Date"])
)

late_customer_history.head(10)

,CustomerID,Purchase_Date,Days_Between,Typical_Interval,Contains_Zero_Day_Product,Days_To_Next_Purchase,Next_Purchase_vs_Rhythm
2,12347.0,2010-12-07 14:57:00,NaN,58.460069,True,49.981250,0.854964
3,12347.0,2011-01-26 14:30:00,49.981250,58.460069,True,70.842361,1.211808
4,12347.0,2011-04-07 10:43:00,70.842361,58.460069,True,63.095833,1.079298
5,12347.0,2011-06-09 13:01:00,63.095833,58.460069,True,53.824306,0.920702
6,12347.0,2011-08-02 08:48:00,53.824306,58.460069,True,90.150694,1.542090
7,12347.0,2011-10-31 12:25:00,90.150694,58.460069,True,37.143750,0.635370
8,12347.0,2011-12-07 15:52:00,37.143750,58.460069,True,NaN,NaN
15,12352.0,2011-02-16 12:33:00,NaN,14.552778,True,13.100000,0.900172
16,12352.0,2011-03-01 14:57:00,13.100000,14.552778,True,0.034722,0.002386
17,12352.0,2011-03-01 15:47:00,0.034722,14.552778,False,0.001389,0.000095


In [65]:
customer_purchase_events["Purchase_Number"] = (
    customer_purchase_events
    .groupby("CustomerID")
    .cumcount()
)

In [66]:
customer_purchase_events["Next_1_Interval"] = (
    customer_purchase_events
    .groupby("CustomerID")["Days_Between"]
    .shift(-1)
)

customer_purchase_events["Next_2_Interval"] = (
    customer_purchase_events
    .groupby("CustomerID")["Days_Between"]
    .shift(-2)
)

customer_purchase_events["Next_3_Interval"] = (
    customer_purchase_events
    .groupby("CustomerID")["Days_Between"]
    .shift(-3)
)

In [67]:
for col in [
    "Next_1_Interval",
    "Next_2_Interval",
    "Next_3_Interval"
]:
    ratio_col = col.replace("_Interval", "_vs_Rhythm")
    
    customer_purchase_events[ratio_col] = (
        customer_purchase_events[col]
        / customer_purchase_events["Typical_Interval"]
    )

In [68]:
late_zero_day_sequence = customer_purchase_events[
    (customer_purchase_events["Contains_Zero_Day_Product"]) &
    (customer_purchase_events["Late_Ratio"] >= 1.25)
][
    [
        "CustomerID",
        "Purchase_Date",
        "Late_Ratio",
        "Next_1_vs_Rhythm",
        "Next_2_vs_Rhythm",
        "Next_3_vs_Rhythm"
    ]
].copy()

In [69]:
late_zero_day_sequence[
    [
        "Late_Ratio",
        "Next_1_vs_Rhythm",
        "Next_2_vs_Rhythm",
        "Next_3_vs_Rhythm"
    ]
].describe().round(2)

,Late_Ratio,Next_1_vs_Rhythm,Next_2_vs_Rhythm,Next_3_vs_Rhythm
count,4194.00,3721.00,3189.00,2732.00
mean,2.70,1.02,1.05,1.02
std,2.86,1.34,1.28,1.28
min,1.25,0.00,0.00,0.00
25%,1.52,0.23,0.25,0.26
50%,1.91,0.75,0.82,0.79
75%,2.82,1.26,1.31,1.29
max,60.45,29.47,21.01,17.69


Customers who were significantly behind their normal purchasing rhythm when they purchased a 0-day product tended to return to approximately their normal purchasing rhythm afterward. The median customer was at 1.95 times their typical purchasing interval before the 0-day purchase, compared with 0.74 times the typical interval for the next purchase. The pattern remained near normal for the following two purchases. This suggests that short-duration products may be associated with re-engagement among customers who have fallen behind their typical purchasing pattern.

#### One Customer's behavior

In [70]:
zero_day_customer_summary = (
    identified[
        identified["StockCode"].isin(zero_day_stock_codes)
    ]
    .groupby("CustomerID")
    .agg(
        Zero_Day_Purchases=("InvoiceNo", "nunique"),
        Zero_Day_Revenue=("Revenue", "sum"),
        Zero_Day_Products=("StockCode", "nunique")
    )
)

customer_example_candidates = (
    customer_summary
    .join(zero_day_customer_summary, how="inner")
    .query("Zero_Day_Purchases >= 2 and Purchases >= 5")
    .sort_values(
        ["Zero_Day_Purchases", "Purchases"],
        ascending=False
    )
)

# customer_example_candidates.head(20)

In [71]:
example_customer = "14156"

customer_history = (
    identified[
        identified["CustomerID"] == example_customer
    ]
    [
        [
            "InvoiceDate",
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "Revenue"
        ]
    ]
    .sort_values("InvoiceDate")
)

customer_history["Zero_Day_Product"] = (
    customer_history["StockCode"].isin(zero_day_stock_codes)
)

customer_history

,InvoiceDate,InvoiceNo,StockCode,Description,Quantity,UnitPrice,Revenue,Zero_Day_Product


In [72]:
customer_regular_products = (
    customer_history[
        ~customer_history["Zero_Day_Product"]
    ]
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Purchases=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        First_Purchase=("InvoiceDate", "min"),
        Last_Purchase=("InvoiceDate", "max")
    )
    .reset_index()
    .sort_values(
        ["Purchases", "Revenue"],
        ascending=[False, False]
    )
)

customer_regular_products.head(10)

,StockCode,Description,Purchases,Units,Revenue,First_Purchase,Last_Purchase


In [73]:
customer_zero_day_products = (
    customer_history[
        customer_history["Zero_Day_Product"]
    ]
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Purchases=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Revenue=("Revenue", "sum"),
        First_Purchase=("InvoiceDate", "min"),
        Last_Purchase=("InvoiceDate", "max")
    )
    .reset_index()
    .sort_values(
        ["Purchases", "Revenue"],
        ascending=[False, False]
    )
)

customer_zero_day_products

,StockCode,Description,Purchases,Units,Revenue,First_Purchase,Last_Purchase


In [74]:
example_customer = "14156"

customer_products = (
    identified[
        identified["CustomerID"] == example_customer
    ]
    .assign(
        Sale_Type=lambda x: np.where(
            x["StockCode"].isin(zero_day_stock_codes),
            "One-Day Sale",
            "Regular Sale"
        )
    )
    [["Description", "Sale_Type"]]
    .drop_duplicates()
    .sort_values("Description", na_position="last")
    .reset_index(drop=True)
)

customer_products

,Description,Sale_Type


In [75]:
customer_products = (
    identified[
        identified["CustomerID"] == example_customer
    ]
    .assign(
        Sale_Type=lambda x: np.where(
            x["StockCode"].isin(zero_day_stock_codes),
            "One-Day Sale",
            "Regular Sale"
        )
    )
    .groupby(["Description", "Sale_Type"], dropna=False)
    .agg(
        Purchases=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
    .sort_values("Description", na_position="last")
)
# pd.set_option('display.max_rows', None)

customer_products

,Description,Sale_Type,Purchases,Units


One-day purchases are often similar to products the customer already buys regularly, but typically differ in some way such as color, design, or style. This suggests that one-day offers may be more effective when they provide a variation of a product the customer already shows an interest in rather than introducing something completely unrelated.